# Week 6 — Multi-table Joins and Complex Aggregations: Geographic Revenue Analysis
## Phase 2b SQL | PORA Academy Cohort 7 — **Exercises**

Thursday's demo walked one pipeline end to end — `orders` → `customers` → `order_payments` — and turned it into the single most useful table Olist has: revenue, order count and delivery speed for every state in Brazil. It showed you that São Paulo takes 40,500 of the 96,478 delivered orders and gets them in 8.8 days, while Alagoas waits 24.5. It also showed you *why* SP is fast and cheap, in one sentence you are now going to prove for yourself: **Olist's sellers are in São Paulo too.**

The four questions below push the same idea in four directions. Q1 stretches the chain to four tables and puts a number on the cost of distance. Q2 stretches it to **five** and asks whether two cities 400 km apart actually want the same things. Q3 flips the geography around — instead of asking where the customers are, it asks where the *money is earned*. Q4 leaves geography behind and tightens the screw on `HAVING` instead.

Each question comes as **three cells**:

1. A **question** with the task and an **Expected** result.
2. A blank `%%sql` answer cell — write your query where it says `-- Your query here`, capturing the result into a variable (e.g. `q1`).
3. A **check cell** (plain Python) — run it after your query. A ✅ means you got it right, and the cell then displays your result table underneath so you can read the answer, not just the tick.

**Do not edit the check cells.** They are the marking scheme. If one fails, the message tells you which number came out wrong and usually why — read it before you start rewriting the query from scratch. Column names matter: the checks look for the exact aliases each question asks for.

Run the setup cell first.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


### Before you start — the chains you are about to build

Three of today's four questions need a join chain longer than anything you have written so far. Write the chain out before you write the SQL; the keys are the whole game, and every one of them is in this table.

| Link | Join on | Why you need it |
|---|---|---|
| `orders o` → `customers c` | `o.customer_id = c.customer_id` | the **buyer's** state (`c.customer_state`) |
| `orders o` → `order_items oi` | `o.order_id = oi.order_id` | the money: `oi.price`, `oi.freight_value` |
| `order_items oi` → `sellers s` | `oi.seller_id = s.seller_id` | the **seller's** state (`s.seller_state`) |
| `order_items oi` → `products p` | `oi.product_id = p.product_id` | `p.product_category_name` (Portuguese) |
| `products p` → `product_category_translation t` | `p.product_category_name = t.product_category_name` | `t.product_category_name_english` |

Two traps worth naming now, because both cost people a whole exercise every cohort:

- **`customer_id`, not `customer_unique_id`.** `customers` has both. `customer_id` is one per order and is the join key; `customer_unique_id` identifies a repeat buyer. Joining on the wrong one returns zero rows and looks like "no data".
- **Two different states live in this database.** `c.customer_state` is where the parcel is *going*; `s.seller_state` is where it is *coming from*. Q1 uses both at once — one in the `WHERE`, one in the `GROUP BY` — and getting them the wrong way round gives you a perfectly plausible, entirely wrong table.

### …and the one rule that decides whether your numbers are real

Every question today aggregates over a join, so before each `SELECT` ask what **one row** of your joined result represents.

| Table | Rows | Distinct `order_id` | Grain |
|---|---|---|---|
| `orders` | 99,441 | 99,441 | one row per order — safe base |
| `customers` | 99,441 | — | one row per order's buyer — safe |
| `order_items` | 112,650 | 98,666 | **many rows per order** (one per line item) |
| `sellers` | 3,095 | — | one row per seller — safe |
| `products` | 32,951 | — | one row per product — safe |

Only one fan-out table appears anywhere today: `order_items`. That is deliberate — none of these four questions joins two many-rows-per-order tables together, so you never need a pre-aggregating `WITH` CTE for safety the way Wednesday's Q3 and Q4 did. But `order_items` still fans out on its own:

- **Counting orders is always `COUNT(DISTINCT ...order_id)`.** An order with three line items becomes three rows; `COUNT(*)` would report it as three orders.
- **Summing and averaging money over `order_items` is correct at line-item grain.** `SUM(price)` is right because every line item really is a separate sale, and `AVG(freight_value)` in Q1 really is "the average freight charge on a line item". You are not fighting the fan-out here — you are choosing it on purpose, which is a different thing, and you should be able to say which one you are doing.

## Question 1 — What does distance from São Paulo actually cost?

The demo left a claim hanging: SP's orders are fast and cheap because SP's *sellers* are next door. Now test it properly. Hold the seller fixed — **only sellers based in SP** — and see what customers in each of the 27 states pay to have that same SP seller's goods shipped to them. Any difference that survives is pure distance, not a different mix of sellers.

This needs all four tables: `orders` → `customers` → `order_items` → `sellers`, chained on the keys in the table above. Filter with `WHERE s.seller_state = 'SP'`, group by `c.customer_state`, and return three columns:

- `customer_state`
- `order_count` — `COUNT(DISTINCT o.order_id)`, not `COUNT(*)` (line items fan out)
- `avg_freight` — `ROUND(AVG(oi.freight_value), 2)`, the average freight charge per line item

Sort with `ORDER BY avg_freight DESC` and return **all** states — no `LIMIT`. The bottom of this list is the interesting end.

**Expected:** 27 rows. Paraíba (`PB`) tops it — 349 orders at an average freight of **R$42.78** — followed by `RR` (R$41.45) and `RO` (R$40.29). At the very bottom sits `SP` itself: **31,502 orders at R$13.20**, the only state under R$19. Rio de Janeiro, next door to SP, pays R$20.47 across 8,457 orders. Read the two ends together: a customer in Paraíba pays **3.2×** what a customer in São Paulo pays to buy from the very same sellers.

In [ ]:
%%sql q1 <<
-- Your query here

In [ ]:
# --- CHECK Q1 — do not edit ---
for col in ['customer_state', 'order_count', 'avg_freight']:
    assert col in q1.columns, f"Q1: missing the '{col}' column — check your SELECT aliases"
assert q1.shape[0] == 27, \
    (f"Q1: expected 27 rows (one per state), got {q1.shape[0]} — drop any LIMIT, and make sure "
     f"the filter is WHERE s.seller_state = 'SP' (the SELLER), not c.customer_state")
assert q1.iloc[0]['customer_state'] == 'PB', \
    (f"Q1: expected 'PB' in the top row, got '{q1.iloc[0]['customer_state']}' — "
     f"ORDER BY avg_freight DESC")
assert int(q1.iloc[0]['order_count']) == 349, \
    (f"Q1: expected PB order_count = 349, got {int(q1.iloc[0]['order_count']):,} — "
     f"use COUNT(DISTINCT o.order_id), not COUNT(*)")
assert abs(float(q1.iloc[0]['avg_freight']) - 42.78) < 0.01, \
    f"Q1: expected PB avg_freight ≈ 42.78, got {q1.iloc[0]['avg_freight']}"
assert list(q1['customer_state'])[:3] == ['PB', 'RR', 'RO'], \
    f"Q1: expected the three costliest states to be PB, RR, RO — got {list(q1['customer_state'])[:3]}"
assert q1.iloc[26]['customer_state'] == 'SP', \
    (f"Q1: expected SP last (cheapest freight), got '{q1.iloc[26]['customer_state']}' — "
     f"sort descending so the cheapest state falls to the bottom")
assert int(q1.iloc[26]['order_count']) == 31502, \
    f"Q1: expected SP order_count = 31,502, got {int(q1.iloc[26]['order_count']):,}"
assert abs(float(q1.iloc[26]['avg_freight']) - 13.20) < 0.01, \
    f"Q1: expected SP avg_freight ≈ 13.20, got {q1.iloc[26]['avg_freight']}"
rj = q1[q1['customer_state'] == 'RJ'].iloc[0]
assert int(rj['order_count']) == 8457 and abs(float(rj['avg_freight']) - 20.47) < 0.01, \
    f"Q1: expected RJ to show 8,457 orders at avg_freight ≈ 20.47, got {int(rj['order_count']):,} at {rj['avg_freight']}"
print("✅ Q1 correct")
q1  # show the result of your query

## Question 2 — Do São Paulo and Rio want the same things?

SP and RJ are Olist's two biggest markets — 40,501 and 12,350 delivered orders — and the merchandising team wants to know whether they can be treated as one market or need separate stock. Answer it with the **top 5 product categories in each state, side by side**.

This is the longest chain of the week: `orders` → `customers` → `order_items` → `products` → `product_category_translation`. Five tables, four joins.

"Top 5 *per state*" is the interesting part. A single `ORDER BY ... LIMIT 5` would give you the top 5 overall, which is not the question — SP is three times RJ's size and would take every slot. Use last week's window function instead. Build it in two CTEs:

```sql
WITH state_cat AS (
    -- one row per (state, category) with its order count
    ...
    WHERE c.customer_state IN ('SP', 'RJ')
    GROUP BY c.customer_state, t.product_category_name_english
),
ranked AS (
    SELECT customer_state, category, order_count,
           ROW_NUMBER() OVER (PARTITION BY customer_state
                              ORDER BY order_count DESC) AS rank_in_state
    FROM state_cat
)
```

`PARTITION BY customer_state` restarts the numbering for each state, so each one gets its own 1–5. Then select from `ranked` with `WHERE rank_in_state <= 5` and finish with `ORDER BY customer_state, rank_in_state`. Four columns, in this order: `customer_state`, `rank_in_state`, `category`, `order_count` (which is `COUNT(DISTINCT o.order_id)`).

**Expected:** 10 rows — RJ first (alphabetical), then SP.

| customer_state | rank | category | order_count |
|---|---|---|---|
| RJ | 1 | bed_bath_table | 1,393 |
| RJ | 2 | health_beauty | 974 |
| RJ | 3 | sports_leisure | 925 |
| RJ | 4 | computers_accessories | 859 |
| RJ | 5 | furniture_decor | 858 |
| SP | 1 | bed_bath_table | 4,416 |
| SP | 2 | health_beauty | 3,789 |
| SP | 3 | sports_leisure | 3,296 |
| SP | 4 | housewares | 2,781 |
| SP | 5 | furniture_decor | 2,724 |

The top three are identical, in the same order. The difference is at ranks 4 and 5: Rio buys `computers_accessories`, São Paulo buys `housewares`. And notice how tight RJ's list is at the bottom — 859 against 858, a single order between fourth and fifth place.

In [ ]:
%%sql q2 <<
-- Your query here

In [ ]:
# --- CHECK Q2 — do not edit ---
for col in ['customer_state', 'rank_in_state', 'category', 'order_count']:
    assert col in q2.columns, f"Q2: missing the '{col}' column — check your SELECT aliases"
assert q2.shape[0] == 10, \
    (f"Q2: expected 10 rows (top 5 in each of 2 states), got {q2.shape[0]} — "
     f"filter WHERE rank_in_state <= 5 after the ROW_NUMBER() CTE")
assert list(q2['customer_state']) == ['RJ'] * 5 + ['SP'] * 5, \
    (f"Q2: expected 5 RJ rows then 5 SP rows — got {list(q2['customer_state'])}; "
     f"finish with ORDER BY customer_state, rank_in_state")
assert [int(r) for r in q2['rank_in_state']] == [1, 2, 3, 4, 5, 1, 2, 3, 4, 5], \
    (f"Q2: ranks should restart at 1 for each state — got {list(q2['rank_in_state'])}; "
     f"did you PARTITION BY customer_state?")
rj_cats = list(q2[q2['customer_state'] == 'RJ']['category'])
sp_cats = list(q2[q2['customer_state'] == 'SP']['category'])
assert rj_cats == ['bed_bath_table', 'health_beauty', 'sports_leisure',
                   'computers_accessories', 'furniture_decor'], \
    f"Q2: unexpected RJ top 5 — got {rj_cats}"
assert sp_cats == ['bed_bath_table', 'health_beauty', 'sports_leisure',
                   'housewares', 'furniture_decor'], \
    f"Q2: unexpected SP top 5 — got {sp_cats}"
rj_counts = [int(n) for n in q2[q2['customer_state'] == 'RJ']['order_count']]
sp_counts = [int(n) for n in q2[q2['customer_state'] == 'SP']['order_count']]
assert rj_counts == [1393, 974, 925, 859, 858], \
    (f"Q2: expected RJ counts [1393, 974, 925, 859, 858], got {rj_counts} — "
     f"use COUNT(DISTINCT o.order_id), not COUNT(*)")
assert sp_counts == [4416, 3789, 3296, 2781, 2724], \
    (f"Q2: expected SP counts [4416, 3789, 3296, 2781, 2724], got {sp_counts} — "
     f"use COUNT(DISTINCT o.order_id), not COUNT(*)")
print("✅ Q2 correct")
q2  # show the result of your query

## Question 3 — Where is the money actually *earned*?

Every geographic query so far has been about the buyer. Turn it around. **GMV** — gross merchandise value, the total the customer hands over, `price + freight_value` — is the number a marketplace reports to its investors, and Olist's board wants it split by the state its *sellers* are in. That is a supply-side question, and answering it takes a much shorter chain than you'd expect.

You do **not** need `orders` or `customers` here. `order_items` already carries both money columns *and* the `seller_id`, so two tables is the whole query: `order_items oi` → `sellers s` on `seller_id`. Group by `s.seller_state` and return three columns:

- `seller_state`
- `order_count` — `COUNT(DISTINCT oi.order_id)`
- `total_gmv` — `ROUND(SUM(oi.price + oi.freight_value), 2)`

`ORDER BY total_gmv DESC`, no `LIMIT` — you want the whole distribution, tail included.

**Expected:** 23 rows (only 23 of Brazil's 27 states have any seller on the platform at all). SP takes the top spot with **70,188 orders and R$10,235,883.88** in GMV; then PR (R$1,458,900.73), MG (R$1,224,159.80) and RJ (R$937,814.12). The bottom of the table is where it gets uncomfortable: the last row is Acre (`AC`) with **one order** worth **R$299.84**.

Add up the column and you get R$15,843,553.24 of GMV in total — which makes São Paulo alone **64.6%** of everything the marketplace sells. Put that next to Q1: the state that sells almost two-thirds of the goods is also the state that gets the cheapest freight, because it is buying from itself.

In [ ]:
%%sql q3 <<
-- Your query here

In [ ]:
# --- CHECK Q3 — do not edit ---
for col in ['seller_state', 'order_count', 'total_gmv']:
    assert col in q3.columns, f"Q3: missing the '{col}' column — check your SELECT aliases"
assert q3.shape[0] == 23, \
    (f"Q3: expected 23 rows (only 23 states host a seller), got {q3.shape[0]} — "
     f"group by s.seller_state and drop any LIMIT")
assert list(q3['seller_state'])[:4] == ['SP', 'PR', 'MG', 'RJ'], \
    f"Q3: expected SP, PR, MG, RJ on top by GMV — got {list(q3['seller_state'])[:4]}"
assert int(q3.iloc[0]['order_count']) == 70188, \
    (f"Q3: expected SP order_count = 70,188, got {int(q3.iloc[0]['order_count']):,} — "
     f"use COUNT(DISTINCT oi.order_id); COUNT(*) counts line items, not orders")
assert abs(float(q3.iloc[0]['total_gmv']) - 10235883.88) < 0.01, \
    (f"Q3: expected SP total_gmv ≈ 10,235,883.88, got {q3.iloc[0]['total_gmv']} — "
     f"8,753,396.21 means you summed price but forgot freight_value")
assert abs(float(q3.iloc[1]['total_gmv']) - 1458900.73) < 0.01, \
    f"Q3: expected PR total_gmv ≈ 1,458,900.73, got {q3.iloc[1]['total_gmv']}"
assert abs(float(q3.iloc[2]['total_gmv']) - 1224159.80) < 0.01, \
    f"Q3: expected MG total_gmv ≈ 1,224,159.80, got {q3.iloc[2]['total_gmv']}"
assert q3.iloc[22]['seller_state'] == 'AC' and int(q3.iloc[22]['order_count']) == 1, \
    (f"Q3: expected AC last with a single order, got "
     f"'{q3.iloc[22]['seller_state']}' with {int(q3.iloc[22]['order_count'])}")
assert abs(float(q3.iloc[22]['total_gmv']) - 299.84) < 0.01, \
    f"Q3: expected AC total_gmv ≈ 299.84, got {q3.iloc[22]['total_gmv']}"
assert abs(float(q3['total_gmv'].sum()) - 15843553.24) < 0.05, \
    f"Q3: the 23 states should sum to ≈ 15,843,553.24 of GMV, got {float(q3['total_gmv'].sum()):,.2f}"
print("✅ Q3 correct")
q3  # show the result of your query

## Question 4 — The premium end of the catalogue

Last one, and it is the shortest — but it is the one where the clause you choose decides whether you get an answer or an error message. Olist wants to know which categories are **premium**: where the average item sells for more than **R$150**. Not which items cost more than R$150 — which *categories average* more than R$150. That distinction is the whole question.

Use Wednesday's three-table category chain: `order_items oi` → `products p` (on `product_id`) → `product_category_translation t` (on `product_category_name`). Group by the English category name and return three columns:

- `category` — `t.product_category_name_english`
- `order_count` — `COUNT(DISTINCT oi.order_id)`
- `avg_price` — `ROUND(AVG(oi.price), 2)`

The R$150 threshold is a condition on an **aggregate**, so it cannot go in `WHERE` — at `WHERE` time the groups don't exist yet and SQLite raises *"misuse of aggregate function AVG()"*. It belongs in `HAVING AVG(oi.price) > 150`, after the `GROUP BY`. Finish with `ORDER BY avg_price DESC`.

**Expected:** 17 rows — 17 of the 71 categories clear the bar. `computers` runs away with it at an average of **R$1,098.34**, but across only **181 orders**; then `small_appliances_home_oven_and_coffee` (R$624.29, 75 orders) and `home_appliances_2` (R$476.12). The last row over the line is `costruction_tools_tools` at R$154.41 — and yes, that misspelling is really in Olist's translation table, which is its own small lesson about trusting source data.

Two cross-checks you can make yourself: `watches_gifts` appears here at **5,624 orders, R$201.14**, exactly as in Wednesday's revenue table, and so does `cool_stuff` at 3,632 orders and R$167.36. If those two rows match, your chain is right. Then look at the shape of the whole list: the categories with the highest average price are almost all the ones with the *fewest* orders. Premium and popular are not the same business.

In [ ]:
%%sql q4 <<
-- Your query here

In [ ]:
# --- CHECK Q4 — do not edit ---
for col in ['category', 'order_count', 'avg_price']:
    assert col in q4.columns, f"Q4: missing the '{col}' column — check your SELECT aliases"
assert q4.shape[0] == 17, \
    (f"Q4: expected 17 categories above R$150, got {q4.shape[0]} — the threshold goes in "
     f"HAVING AVG(oi.price) > 150, after the GROUP BY, not in WHERE")
assert q4.iloc[0]['category'] == 'computers', \
    f"Q4: expected 'computers' first, got '{q4.iloc[0]['category']}' — ORDER BY avg_price DESC"
assert abs(float(q4.iloc[0]['avg_price']) - 1098.34) < 0.01, \
    f"Q4: expected computers avg_price ≈ 1,098.34, got {q4.iloc[0]['avg_price']}"
assert int(q4.iloc[0]['order_count']) == 181, \
    (f"Q4: expected computers order_count = 181, got {int(q4.iloc[0]['order_count']):,} — "
     f"use COUNT(DISTINCT oi.order_id), not COUNT(*)")
assert list(q4['category'])[:3] == ['computers', 'small_appliances_home_oven_and_coffee',
                                    'home_appliances_2'], \
    f"Q4: unexpected top 3 — got {list(q4['category'])[:3]}"
assert abs(float(q4.iloc[1]['avg_price']) - 624.29) < 0.01, \
    f"Q4: expected small_appliances_home_oven_and_coffee avg_price ≈ 624.29, got {q4.iloc[1]['avg_price']}"
assert q4.iloc[16]['category'] == 'costruction_tools_tools', \
    (f"Q4: expected 'costruction_tools_tools' (sic) as the last row above the bar, got "
     f"'{q4.iloc[16]['category']}'")
assert abs(float(q4.iloc[16]['avg_price']) - 154.41) < 0.01, \
    f"Q4: expected costruction_tools_tools avg_price ≈ 154.41, got {q4.iloc[16]['avg_price']}"
wg = q4[q4['category'] == 'watches_gifts'].iloc[0]
assert int(wg['order_count']) == 5624 and abs(float(wg['avg_price']) - 201.14) < 0.01, \
    (f"Q4: watches_gifts should match Wednesday's table — 5,624 orders at R$201.14, got "
     f"{int(wg['order_count']):,} at {wg['avg_price']}")
cs = q4[q4['category'] == 'cool_stuff'].iloc[0]
assert int(cs['order_count']) == 3632 and abs(float(cs['avg_price']) - 167.36) < 0.01, \
    (f"Q4: cool_stuff should match Wednesday's table — 3,632 orders at R$167.36, got "
     f"{int(cs['order_count']):,} at {cs['avg_price']}")
print("✅ Q4 correct")
q4  # show the result of your query

## When all four are green

Four queries, one story. Q1 measured the cost of distance from SP's sellers (R$13.20 at home against R$42.78 in Paraíba). Q3 showed why that matters so much: **64.6%** of everything Olist sells ships from São Paulo. Q2 found that Rio and São Paulo want the same three things and diverge only at ranks 4 and 5. Q4 stepped away from the map to separate the premium categories from the popular ones — and found they are almost never the same category.

Three habits to carry out of Week 6:

- **Draw the chain before you type it.** Every question above was a list of tables and the key joining each pair. Once that list is on paper the SQL is mechanical; without it, four joins is where people start guessing.
- **Name which state you mean.** `customer_state` and `seller_state` are both in this database and both are called "state" in conversation. Q1 needed both at once. Whenever a stakeholder says "revenue by state", your first question back is *whose* state.
- **Know why each aggregate is safe.** Today `COUNT(DISTINCT order_id)` was mandatory everywhere, while `SUM(price + freight_value)` and `AVG(freight_value)` over `order_items` were correct at line-item grain on purpose. Being able to say which of those two things you are doing — and why — is the actual skill this week was teaching.

If you drafted any of these with DeepSeek, the rule from the demo stands: a plausible table is not a correct table. The green ✅ is the only evidence that counts.

---
**Coming up next week:** you'll take these multi-table pipelines and turn them into reusable, readable analysis with CTEs and views — the tools that stop a five-table query from becoming unmaintainable.